# QLoRA Fine-Tuning for Multimodal Alignment of Voxtral with GLaDOS Persona

In [1]:
import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import (
    AutoProcessor,
    VoxtralForConditionalGeneration,
    BitsAndBytesConfig,
    TrainingArguments,
    DataCollatorForSeq2Seq
)
from transformers.trainer_utils import get_last_checkpoint
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
import os
import gc
import json
import wandb
from tqdm.auto import tqdm
import warnings

warnings.filterwarnings("ignore", category=UserWarning, module="bitsandbytes")
HYPERTUNE = True # Set to True to run hyperparameter sweep, False for single training run

os.environ["WANDB_PROJECT"] = "Voxtral-GLaDOS-Multimodal"
os.environ["WANDB_LOG_MODEL"] = "false"
os.environ["WANDB_NOTEBOOK_NAME"] = "qlora_finetune.ipynb"
wandb.login()
device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = "mistralai/Voxtral-Mini-3B-2507"
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

processor = AutoProcessor.from_pretrained(model_id)
processor.tokenizer.padding_side = "left"
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token
    processor.tokenizer.pad_token_id = processor.tokenizer.eos_token_id

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4", # Highly optimized for speed/accuracy
    bnb_4bit_use_double_quant=True, # Saves extra memory at no speed cost
    bnb_4bit_compute_dtype=compute_dtype
)

model = VoxtralForConditionalGeneration.from_pretrained(
            model_id,
            quantization_config=bnb_config,
            attn_implementation="flash_attention_2",
            device_map=device
        )
print(model)
del model

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/vito/.netrc.
wandb: Currently logged in as: vitolus (vitolus-universit-ca-foscari-venezia) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/762 [00:00<?, ?it/s]

VoxtralForConditionalGeneration(
  (audio_tower): VoxtralEncoder(
    (conv1): Conv1d(128, 1280, kernel_size=(3,), stride=(1,), padding=(1,))
    (conv2): Conv1d(1280, 1280, kernel_size=(3,), stride=(2,), padding=(1,))
    (embed_positions): Embedding(1500, 1280)
    (layers): ModuleList(
      (0-31): 32 x VoxtralEncoderLayer(
        (self_attn): VoxtralAttention(
          (k_proj): Linear4bit(in_features=1280, out_features=1280, bias=False)
          (v_proj): Linear4bit(in_features=1280, out_features=1280, bias=True)
          (q_proj): Linear4bit(in_features=1280, out_features=1280, bias=True)
          (out_proj): Linear4bit(in_features=1280, out_features=1280, bias=True)
        )
        (self_attn_layer_norm): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
        (activation_fn): GELUActivation()
        (fc1): Linear4bit(in_features=1280, out_features=5120, bias=True)
        (fc2): Linear4bit(in_features=5120, out_features=1280, bias=True)
        (final_layer_norm

#### Dataset Formatting for Multimodal SFT

In [2]:
df = pd.read_csv("./data/combined_multimodal_dataset_train.csv")
# Extract all unique user commands
unique_commands = df['User_Command'].unique().tolist()
print(f"Total Unique Commands: {len(unique_commands)}")

def format_dataframe_to_dataset(df, audio_dir="./data/synthesized_train/"):
    dataset_dict = {"messages": []}
    missing_files = 0
    for _, row in df.iterrows():
        full_audio_path = os.path.join(audio_dir, row["Audio_File"])
        # Validation check to prevent the trainer from crashing mid-epoch
        if not os.path.exists(full_audio_path):
            missing_files += 1
            print(f"Warning: Skipping {row['Audio_File']} - File not found at {full_audio_path}")
            continue
        target_output = f"{row['Assistant_Payload']}\n\n{row['Target_GLaDOS_Response']}{processor.tokenizer.eos_token}"
        conversation = [
            {"role": "user", "content": [{"type": "audio", "path": full_audio_path}]}, # The processor will handle loading and feature extraction during training
            {"role": "assistant", "content": [{"type": "text", "text": target_output}]},
            {"role": "user", "content": [{"type": "text", "text": "DUMMY_STOP"}]} # A dummy user turn to satisfy the Mistral Serving Validator
        ]
        dataset_dict["messages"].append(conversation)
    if missing_files > 0:
        print(f"Warning: Skipped {missing_files} rows due to missing audio files.")
    # No need to cast to datasets.Audio(), saving massive amounts of RAM
    return Dataset.from_dict(dataset_dict)

def create_datasets(df, commands_list, test_size=0.1):
    train_cmds, eval_cmds = train_test_split(commands_list, test_size=test_size, random_state=42)
    # Filter the dataframe so all variations of a command stay strictly together
    df_train = df[df['User_Command'].isin(train_cmds)]
    df_eval = df[df['User_Command'].isin(eval_cmds)]
    print(f"Train rows: {len(df_train)} | Eval rows: {len(df_eval)}")
    train_ds = format_dataframe_to_dataset(df_train).shuffle(seed=42)
    eval_ds = format_dataframe_to_dataset(df_eval)
    return train_ds, eval_ds

Total Unique Commands: 10184


In [3]:
def voxtral_collate_fn(batch):
    conversations = [item["messages"] for item in batch]
    # Let the processor handle the audio loading, padding, and tokenization.
    inputs = processor.apply_chat_template(
        conversations,
        tokenize=True,
        return_dict=True,
        processor_kwargs={"padding": True, "return_tensors": "pt"}
    )
    input_ids = inputs["input_ids"]
    attention_mask = inputs["attention_mask"]
    labels = input_ids.clone()
    inst_token_ids = processor.tokenizer.encode("[/INST]", add_special_tokens=False)
    seq_len = len(inst_token_ids)
    inst_seq = torch.tensor(inst_token_ids, device=labels.device)
    for i in range(labels.shape[0]):
        # Mask User Input and Left Padding via Sequence Matching
        # .unfold creates a sliding window of size `seq_len` to check for the exact token sequence
        matches = (labels[i].unfold(0, seq_len, 1) == inst_seq).all(dim=1)
        inst_indices = matches.nonzero(as_tuple=True)[0]
        if len(inst_indices) > 0:
            # Shift the index to the END of the [/INST] sequence
            first_inst_end_idx = inst_indices[0] + seq_len - 1
            # Mask everything up to and including the [/INST] sequence
            labels[i, :first_inst_end_idx + 1] = -100
        # Mask right-side dummy padding
        eos_indices = (labels[i] == processor.tokenizer.eos_token_id).nonzero(as_tuple=True)[0]
        if len(eos_indices) > 0:
            # The first EOS belongs to our Assistant response
            target_eos_idx = eos_indices[0]
            # Safely mask everything after the target EOS token
            labels[i, target_eos_idx + 1:] = -100
            # Clean up attention mask and input ids for the dummy turn
            attention_mask[i, target_eos_idx + 1:] = 0
            input_ids[i, target_eos_idx + 1:] = processor.tokenizer.pad_token_id
    inputs["input_ids"] = input_ids
    inputs["attention_mask"] = attention_mask
    inputs["labels"] = labels
    return inputs

#### Hyperpatameters tuning

In [ ]:
def create_sweep_func(train_dataset, eval_dataset):
    def train_sweep():
        wandb.init()
        config = wandb.config
        # Reload Base Model (clears old adapters from memory)
        model = VoxtralForConditionalGeneration.from_pretrained(
            model_id,
            quantization_config=bnb_config,
            attn_implementation="flash_attention_2",
            device_map=device
        )

        model = prepare_model_for_kbit_training(model)
        model.get_input_embeddings().to(compute_dtype)
        if hasattr(model, "get_output_embeddings") and model.get_output_embeddings() is not None:
            model.get_output_embeddings().to(compute_dtype)
        if hasattr(model, "audio_encoder"):
            model.audio_encoder.to(compute_dtype)
        model.config.pad_token_id = processor.tokenizer.pad_token_id
        model.config.eos_token_id = processor.tokenizer.eos_token_id
        model.config.bos_token_id = processor.tokenizer.bos_token_id

        # Dynamic LoRA Config from Sweep
        lora_config = LoraConfig(
            r=config.lora_r,
            lora_alpha=config.lora_alpha,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
            lora_dropout=config.lora_dropout,
            bias="none",
            task_type="CAUSAL_LM"
        )
        # Dynamic Training Args
        training_args = SFTConfig(
            output_dir="./models/voxtral-sweep",
            per_device_train_batch_size=1,
            per_device_eval_batch_size=4,
            eval_strategy="steps",
            eval_steps=25,
            save_strategy="no",
            load_best_model_at_end=False,
            gradient_checkpointing=True,
            gradient_checkpointing_kwargs={"use_reentrant": False},
            gradient_accumulation_steps=4,
            dataloader_num_workers=4,
            dataloader_pin_memory=True,
            learning_rate=config.learning_rate,
            max_steps=100,
            logging_steps=10,
            optim="paged_adamw_8bit",
            bf16=torch.cuda.is_bf16_supported(),
            fp16=not torch.cuda.is_bf16_supported(),
            remove_unused_columns=False,
            dataset_kwargs={"skip_prepare_dataset": True},
            report_to="wandb",
            # loss_type="chunked_nll",
            neftune_noise_alpha=5,
            lr_scheduler_type="cosine",
            warmup_ratio=0.03,
            max_length=2048
        )
        # Initialize Trainer
        trainer = SFTTrainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=eval_dataset,
            data_collator=voxtral_collate_fn,
            peft_config=lora_config,
        )

        trainer.train()
        del model, trainer
        gc.collect()
        torch.cuda.empty_cache()
    return train_sweep

if HYPERTUNE:
    subset_commands = unique_commands[:1000]
    train_dataset, eval_dataset = create_datasets(df, subset_commands)
    eval_dataset = eval_dataset.select(range(300))
    project = "Voxtral-GLaDOS-Multimodal"

    sweep_config = {
        'method': 'bayes', # Bayesian optimization (smarter than random search)
        'metric': {'name': 'eval/loss', 'goal': 'minimize'},
        'early_terminate': {
            'type': 'hyperband',
            'min_iter': 3, # Minimum number of iterations to run
            'eta': 2 # Aggressiveness of early stopping (higher = more aggressive). Hyperband will stop poorly performing runs early based on intermediate results, allowing more resources for promising configurations.
        },
        'parameters': {
            'learning_rate': {'distribution': 'log_uniform_values', 'min': 1e-5, 'max': 5e-4},
            'lora_r': {'values': [8, 16, 32]}, # Rank of the adapters
            'lora_alpha': {'values': [16, 32, 64]}, # Scaling factor
            'lora_dropout': {'values': [0.05, 0.1]} # Dropout for regularization
        }
    }
    api = wandb.Api()
    entity = api.default_entity if api.default_entity else wandb.setup().settings.entity
    sweep_id = wandb.sweep(sweep_config, project=project)
    # Execute the sweep (Run 5 different combinations)
    wandb.agent(sweep_id, create_sweep_func(train_dataset, eval_dataset), count=5)
    print("\nSweep complete! Fetching the best parameters from W&B cloud...")
    best_params = api.sweep(f"{entity}/{project}/{sweep_id}").best_run().config
    print(f"Best Parameters Found: {best_params}")
    with open("./models/best_sweep_params.json", "w") as f:
        json.dump(best_params, f, indent=4)
    del train_dataset, eval_dataset, api, entity, sweep_id
    gc.collect()
    print("Generating complete datasets for final fine-tuning...")
    train_dataset, eval_dataset = create_datasets(df, unique_commands)
else:
    print(f"Skipping sweep...")
    params_path = "./models/best_sweep_params.json"
    if os.path.exists(params_path):
        print(f"Found existing configuration at {params_path}")
        with open(params_path, "r") as f:
            best_params = json.load(f)
    else:
        best_params = {
        'learning_rate': 2e-4,
        'lora_r': 16,
        'lora_alpha': 32,
        'lora_dropout': 0.1
        }
        print(f"Using default parameters: {best_params}")
    print("Generating complete datasets for final fine-tuning...")
    train_dataset, eval_dataset = create_datasets(df, unique_commands)
del df, unique_commands
gc.collect()
torch.cuda.empty_cache()

Train rows: 6588 | Eval rows: 711
Create sweep with ID: tfz9sapy
Sweep URL: https://wandb.ai/vitolus-universit-ca-foscari-venezia/Voxtral-GLaDOS-Multimodal/sweeps/tfz9sapy


wandb: Agent Starting Run: cq7zjtt1 with config:
wandb: 	learning_rate: 3.0940727324004265e-05
wandb: 	lora_alpha: 32
wandb: 	lora_dropout: 0.1
wandb: 	lora_r: 8
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/vito/.netrc.


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/762 [00:00<?, ?it/s]

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
wandb: WARNING Config item 'learning_rate' was locked by 'sweep' (ignored update).
[transformers] Casting fp32 inputs back to torch.bfloat16 for flash-attn compatibility.


...

wandb: Agent Starting Run: 9se33g2s with config:
wandb: 	learning_rate: 0.0002393067658697793
wandb: 	lora_alpha: 64
wandb: 	lora_dropout: 0.05
wandb: 	lora_r: 32
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/vito/.netrc.


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/762 [00:00<?, ?it/s]

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
wandb: WARNING Config item 'learning_rate' was locked by 'sweep' (ignored update).


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
25,6.282400,6.283512,6.361242,42984.000000,0.134796
50,6.166121,6.259232,6.361408,86006.000000,0.177478


#### Supervised Fine-Tuning with TRL's SFTTrainer

In [ ]:
print(f"Loading {model_id} in 4 bit precision...")
output_dir = "./models/voxtral-glados-sft"
run_id_file = os.path.join(output_dir, "wandb_run_id.txt")
last_checkpoint = None
wandb_run_id = None
if os.path.exists(output_dir):
    last_checkpoint = get_last_checkpoint(output_dir)
    if last_checkpoint and os.path.exists(run_id_file):
        with open(run_id_file, "r") as f:
            wandb_run_id = f.read().strip()

if last_checkpoint and wandb_run_id:
    print(f"Resuming W&B run: {wandb_run_id}...")
    wandb.init(project="Voxtral-GLaDOS-Multimodal", id=wandb_run_id, resume="must")
else:
    print("Starting a new W&B run...")
    wandb_run_id = wandb.util.generate_id()
    os.makedirs(output_dir, exist_ok=True)
    with open(run_id_file, "w") as f:
        f.write(wandb_run_id)
    wandb.init(project="Voxtral-GLaDOS-Multimodal", name="voxtral-GLaDOS", id=wandb_run_id, resume="allow")

model = VoxtralForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    attn_implementation="flash_attention_2",
    device_map=device
)

# Prepare model for gradient training
model = prepare_model_for_kbit_training(model)
# Revert the text embeddings back to bfloat16/float16 to match the Audio Encoder
model.get_input_embeddings().to(compute_dtype)
# Also ensure the output layer matches
if hasattr(model, "get_output_embeddings") and model.get_output_embeddings() is not None:
    model.get_output_embeddings().to(compute_dtype)
# It is also good practice to ensure the audio encoder didn't get accidentally cast to float32
if hasattr(model, "audio_encoder"):
    model.audio_encoder.to(compute_dtype)
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.eos_token_id = processor.tokenizer.eos_token_id
model.config.bos_token_id = processor.tokenizer.bos_token_id

lora_config = LoraConfig(
    r=best_params['lora_r'],
    lora_alpha=best_params['lora_alpha'],
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=best_params['lora_dropout'],
    bias="none",
    task_type="CAUSAL_LM"
)

training_args = SFTConfig(
    output_dir=output_dir,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=4,
    eval_strategy="steps",
    eval_steps=800,
    save_strategy="steps",
    save_steps=800,
    save_total_limit=3,
    load_best_model_at_end=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    gradient_accumulation_steps=16,
    dataloader_num_workers=4,
    dataloader_pin_memory=True,
    learning_rate=best_params['learning_rate'],
    logging_steps=10,
    num_train_epochs=5,
    optim="paged_adamw_8bit", # paged_adamw_8bit use ram if vram is saturated (paging)
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    remove_unused_columns=False, # Crucial so the collator receives the dicts
    dataset_kwargs={"skip_prepare_dataset": True},
    report_to="wandb",
    # loss_type="chunked_nll", # Accumulate loss in smaller chunks to prevent overflow with long sequences
    neftune_noise_alpha=5, # Add a small amount of noise to the activations during training to improve generalization
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    max_length=2048
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=voxtral_collate_fn,
    processing_class=processor,
    peft_config=lora_config
)
trainer.model.print_trainable_parameters()

In [ ]:
print("Initiating QLoRA Multimodal Alignment...")
if last_checkpoint is not None:
    print(f"Resuming training from {last_checkpoint}...")
    trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    print("Starting a new training run...")
    trainer.train()

# Save the final adapter weights
trainer.save_model(os.path.join(output_dir, "final_adapters"))
wandb.finish()
print("Training complete. Adapters saved.")